# 专题：`if __name__ == '__main__'`、Python 文件调用与算法项目实战

这不是一本 Python 语法大全，而是一份面向 **LeetCode、秋招笔试、技术面试和算法工程项目** 的实用手册。

学完以后，你应该能够：

1. 逐字解释 `if __name__ == '__main__':`。
2. 判断一个 `.py` 文件是被直接运行，还是被另一个文件导入。
3. 掌握 Python 程序在日常开发中主要的运行、导入、输入和输出方式。
4. 在 LeetCode 核心代码模式与 ACM 完整程序模式之间转换。
5. 看懂一个规范的算法工程项目目录，知道每个文件负责什么。
6. 写出“算法逻辑可测试、输入输出可替换、程序入口清晰”的代码。

> 阅读建议：第一次先完成 A、B、D、F；准备笔试时重点复习 E；准备项目面试时重点复习 G、H。


## 学习地图

| 模块 | 解决的问题 | 优先级 |
|---|---|---|
| A | `.py` 文件、模块、脚本、包分别是什么 | 必学 |
| B | `__name__` 和 `__main__` 到底在判断什么 | 必学 |
| C | Python 文件有哪些常见运行与调用方式 | 必学 |
| D | `import` 如何工作，为什么会导入失败 | 必学 |
| E | 笔试和面试如何输入输出 | 秋招必学 |
| F | `main()`、`solve()`、返回值和打印如何分工 | 必学 |
| G | 优秀算法工程项目应该怎样组织 | 项目面试必学 |
| H | 一个可运行、可测试的小型项目长什么样 | 进阶 |
| I | 练习、答案与最终速查表 | 必做 |


# A. 先认识 Python 文件

## A01. `.py` 文件是什么

一个 `.py` 文件可以有两种常见身份：

- **脚本（script）**：你直接让 Python 执行它，例如 `python app.py`。
- **模块（module）**：它被其他 Python 代码导入，例如 `import app`。

同一个文件可以同时具备两种身份。`if __name__ == '__main__':` 就是帮助它区分这两种身份。


In [ ]:
# 这是一个最小 Python 文件能包含的内容
PI = 3.14159                       # 变量 / 常量

def circle_area(radius):           # 函数
    return PI * radius * radius

class Circle:                      # 类
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return circle_area(self.radius)

print(circle_area(2))              # 顶层可执行语句


### 什么叫“顶层代码”

没有缩进在函数或类里面、Python 读取文件时就会处理的代码，叫 **顶层代码（top-level code）**。

- `PI = 3.14159`：顶层代码，会创建变量。
- `def circle_area(...):`：顶层代码，会创建函数对象；函数体暂时不执行。
- `class Circle:`：顶层代码，会创建类。
- `print(...)`：顶层代码，会立刻打印。

关键结论：**直接运行会执行顶层代码，第一次导入也会执行顶层代码。** 所以不要在可复用模块的顶层随意写读输入、跑训练、访问数据库等操作。


## A02. 文件、模块、包、项目的区别

| 名称 | 通俗理解 | 示例 |
|---|---|---|
| Python 文件 | 硬盘上的一个文件 | `metrics.py` |
| 模块 | 被 Python 加载后的一个代码单元 | `import metrics` 后的 `metrics` |
| 包 | 用目录组织的一组模块 | `my_project/algorithms/` |
| 项目 | 源码、测试、配置、文档、数据说明等完整集合 | 一个 Git 仓库 |

初学阶段可以记成：**一个 `.py` 文件通常对应一个模块，多个有关模块放进一个包，多个包和工程文件组成项目。**


# B. 彻底理解 `if __name__ == '__main__':`

## B01. 先逐个拆开

```python
if __name__ == '__main__':
    main()
```

- `if`：如果条件成立，就执行缩进代码。
- `__name__`：Python 自动提供的特殊变量，表示“当前模块叫什么”。
- `==`：比较左右两边的值是否相等。
- `'__main__'`：一个特殊字符串，表示当前模块是程序最先执行的入口。
- `main()`：普通函数名，只是一种约定，不是 Python 关键字。

整句人话：**如果当前文件是程序入口，就调用 `main()`；如果它只是被导入，就不要调用。**


In [ ]:
# 在 Jupyter 中，当前顶层环境通常也叫 __main__
print("当前 __name__ 的值：", __name__)
print("是否处于顶层环境：", __name__ == "__main__")


## B02. 两种运行方式决定两种 `__name__`

假设文件叫 `calculator.py`：

| 使用方式 | `calculator.py` 内部的 `__name__` | 入口代码是否执行 |
|---|---|---|
| `python calculator.py` | `'__main__'` | 是 |
| `import calculator` | `'calculator'` | 否 |
| `python -m calculator` | `'__main__'` | 是 |

注意：`__main__` 不是文件名。它是 Python 给“本次程序入口”临时起的特殊模块名。


In [ ]:
# 在临时目录中创建模块，亲眼观察“直接运行”和“导入”的区别
import subprocess
import shutil
import sys
import uuid
from pathlib import Path

source = '''
print("模块被加载，__name__ =", __name__)

def add(a, b):
    return a + b

if __name__ == "__main__":
    print("入口代码执行：", add(2, 3))
'''

folder = Path(f".notebook_demo_{uuid.uuid4().hex}")
folder.mkdir()
try:
    path = folder / "calculator.py"
    path.write_text(source, encoding="utf-8")

    direct = subprocess.run(
        [sys.executable, str(path)], capture_output=True, text=True, check=True
    )
    imported = subprocess.run(
        [sys.executable, "-c", "import calculator; print(calculator.add(10, 20))"],
        cwd=folder, capture_output=True, text=True, check=True
    )
finally:
    shutil.rmtree(folder)

print("【直接运行】")
print(direct.stdout.strip())
print("\n【作为模块导入】")
print(imported.stdout.strip())


### 观察结果

两种方式都会加载模块，所以第一行顶层 `print` 都执行了；但是只有直接运行时，`__name__ == '__main__'` 才成立。

这也解释了一个常见误区：

> `if __name__ == '__main__':` 不能阻止整个文件被执行。导入时，顶层定义和顶层语句仍会运行；它只能保护缩进在这个 `if` 下面的入口操作。


## B03. 为什么需要它

没有入口保护的写法：

```python
def two_sum(nums, target):
    ...

nums = list(map(int, input().split()))  # 导入时也会等待输入
target = int(input())
print(two_sum(nums, target))
```

另一个文件只想导入 `two_sum`，却会被迫输入数据。这叫 **导入副作用（import side effect）**。

推荐写法：

```python
def two_sum(nums, target):
    ...

def main():
    nums = list(map(int, input().split()))
    target = int(input())
    print(two_sum(nums, target))

if __name__ == '__main__':
    main()
```


## B04. `main()` 不是必须，但非常值得使用

下面两种都能运行：

```python
if __name__ == '__main__':
    print('开始')
```

```python
def main():
    print('开始')

if __name__ == '__main__':
    main()
```

第二种更好，因为：

- 入口逻辑有名字，结构清楚。
- `main()` 里的临时变量不会污染整个模块的全局命名空间。
- 测试时可以直接调用 `main()`。
- 将来改成命令行工具时更方便。


In [ ]:
def calculate_total(prices):
    '''纯逻辑函数：接收数据，返回结果，不负责输入输出。'''
    return sum(prices)


def main():
    '''程序入口：准备输入，调用逻辑，展示输出。'''
    prices = [12, 8, 20]
    total = calculate_total(prices)
    print("总价：", total)
    return 0


if __name__ == "__main__":
    main()


## B05. `return 0` 与 `sys.exit(main())`

工程命令行程序常写成：

```python
import sys

def main() -> int:
    # 成功返回 0，失败返回非 0
    return 0

if __name__ == '__main__':
    raise SystemExit(main())
```

- 函数中的 `return 0` 只是把 `0` 返回给调用者。
- `SystemExit(0)` 或 `sys.exit(0)` 会把状态码交给操作系统。
- 通常 `0` 表示成功，非 `0` 表示失败。

刷题时不需要写退出码；制作真正的命令行工具时再使用。


## B06. Jupyter 中为什么它也常常成立

Notebook 内核通常把当前交互环境当成顶层环境，所以代码格中的 `__name__` 往往是 `'__main__'`。因此把一整份脚本粘进 notebook，入口代码可能立即运行。

但 notebook 不是普通 `.py` 文件：

- 代码格可以乱序运行。
- 变量状态会留在内核里。
- 它适合探索和展示，不适合作为大型项目的核心源码。

推荐做法：可复用函数放在 `src/` 里的 `.py` 文件；notebook 负责实验、图表和讲解，通过 `import` 调用源码。


# C. Python 程序的主要运行与调用方式

“调用 Python 文件”可能指三件不同的事：

1. **启动程序**：让一个文件成为入口。
2. **导入代码**：复用另一个文件里的函数、类或常量。
3. **启动另一个进程**：当前程序调用外部 Python 程序。

下面覆盖日常开发、刷题和算法工程中主要且实用的方式。C API 嵌入解释器等极少见场景不属于秋招准备范围。


## C01. 方式总表

| 方式 | 示例 | 适合场景 | 是否产生新进程 |
|---|---|---|---|
| 交互解释器 | `python` / `py` | 临时试一行语法 | 是 |
| 运行脚本 | `python app.py` | 单文件程序、ACM | 是 |
| 运行模块 | `python -m package.module` | 包内入口、避免路径混乱 | 是 |
| 运行包 | `python -m package` | 包含 `__main__.py` 的应用 | 是 |
| 执行短代码 | `python -c "print(1)"` | 命令行临时操作 | 是 |
| 标准输入喂代码 | `python < script.py` | shell/自动化，较少手写 | 是 |
| 导入模块 | `import utils` | 同一程序中复用代码 | 否 |
| 导入成员 | `from utils import add` | 明确复用某个名称 | 否 |
| Notebook | Jupyter 代码格 | 探索、教学、实验 | 使用已有内核 |
| IDE 运行按钮 | VS Code/PyCharm Run | 本地开发，本质仍是启动解释器 | 通常是 |
| 安装后的命令 | `risk-train --config ...` | 正式 CLI 工具 | 是 |
| 子进程调用 | `subprocess.run(...)` | 调用独立程序 | 是 |


## C02. 直接运行脚本：`python file.py`

```powershell
python solution.py
```

Python 会把 `solution.py` 当入口，文件内 `__name__` 为 `'__main__'`。脚本后面的参数放进 `sys.argv`：

```powershell
python train.py --config configs/base.yaml --seed 42
```

Windows 有时也可以使用 `py train.py`；为了团队命令统一，项目文档应明确使用哪一个解释器或环境。


In [ ]:
# sys.argv 是字符串列表：第 0 项通常是程序名，后面才是用户参数
import sys

fake_argv = ["train.py", "--seed", "42"]
print("模拟 argv：", fake_argv)
print("程序名：", fake_argv[0])
print("参数：", fake_argv[1:])


## C03. 运行模块：`python -m package.module`

```powershell
python -m my_project.cli
```

`-m` 的意思是：按照 Python 的模块搜索规则找到模块，再把它作为入口运行。

项目代码中通常优先使用 `-m`，因为它更尊重包结构。例如：

```powershell
python -m pytest
python -m my_project.train --config configs/base.toml
```

它和 `python path/to/cli.py` 最大的差别不是代码内容，而是 **Python 如何确定包关系和导入路径**。


## C04. 运行一个包：`__main__.py`

目录：

```text
text_tool/
├── __init__.py
├── __main__.py
└── cleaner.py
```

执行：

```powershell
python -m text_tool
```

Python 会运行 `text_tool/__main__.py`。规范做法是让这个文件保持很薄：

```python
from .cli import main

raise SystemExit(main())
```

真正逻辑放在其他模块，方便导入和测试。


## C05. 导入模块的常见写法

```python
import math
math.sqrt(9)

import numpy as np
np.array([1, 2, 3])

from collections import Counter
Counter('banana')

from my_project.metrics import accuracy
accuracy(y_true, y_pred)
```

推荐顺序：标准库、第三方库、本项目代码，三组之间空一行。

避免：

```python
from utils import *
```

因为读代码的人不知道名称来自哪里，也容易发生同名覆盖。


In [ ]:
import math
from collections import Counter

print(math.sqrt(81))
print(Counter("banana"))


## C06. `python -c`、管道和重定向

临时执行一小段代码：

```powershell
python -c "print(sum([1, 2, 3]))"
```

把文件内容作为程序的标准输入：

```powershell
Get-Content input.txt | python solution.py
```

把输出保存到文件：

```powershell
python solution.py < input.txt > output.txt
```

这些方式常见于自动评测、脚本和 CI。初学时理解“标准输入流入程序，标准输出流出程序”即可。


## C07. 从 Python 调用另一个程序：`subprocess`

只有当目标确实是一个独立程序时才使用子进程。复用同项目里的函数时，优先 `import`，因为更快、更容易测试，也不用解析文本输出。


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-c", "print(6 * 7)"],
    capture_output=True,
    text=True,
    check=True,
)
print("子进程输出：", result.stdout.strip())


## C08. 安装后的命令行入口

现代 Python 项目可在 `pyproject.toml` 中声明：

```toml
[project.scripts]
risk-train = "risk_project.cli:main"
```

安装项目后，终端中就能运行：

```powershell
risk-train --config configs/base.toml
```

它本质上会导入 `risk_project.cli` 并调用 `main()`。这是正式工具比“让用户找到某个深层脚本路径”更友好的方式。


# D. `import` 到底做了什么

## D01. 第一次导入的四个关键动作

以 `import calculator` 为例，可以先简化理解为：

1. Python 在模块搜索路径中寻找 `calculator`。
2. 创建模块对象。
3. 执行模块顶层代码，建立其中的函数、类和变量。
4. 把模块对象绑定到当前文件的名字 `calculator`。

所以 `import` 不是“把文字复制过来”，而是加载一个模块对象。


In [ ]:
import math

print("模块对象：", math)
print("模块名称：", math.__name__)
print("模块里的函数：", math.sqrt)
print("调用函数：", math.sqrt(16))


## D02. 模块搜索路径 `sys.path`

Python 通常会在这些位置寻找模块：

- 当前入口脚本所在目录或当前工作目录。
- 环境变量 `PYTHONPATH` 指定的位置。
- 当前虚拟环境的标准库和 `site-packages`。

可以观察 `sys.path`，但不要把“到处 `sys.path.append(...)`”当成正式项目方案。更好的办法是正确组织包，并将项目以可编辑模式安装。


In [ ]:
import sys

print("当前解释器：", sys.executable)
print("模块搜索路径前 3 项：")
for item in sys.path[:3]:
    print("-", item)


## D03. `__init__.py` 的作用

传统、清晰的 Python 包目录会包含 `__init__.py`：

```text
algorithms/
├── __init__.py
├── arrays.py
└── linked_list.py
```

它可以是空文件。它主要表示“这个目录是普通 Python 包”，也可以谨慎地暴露稳定接口：

```python
from .arrays import two_sum

__all__ = ['two_sum']
```

不要在 `__init__.py` 中放大量耗时初始化，否则每次导入包都可能变慢或产生副作用。


## D04. 绝对导入与相对导入

假设当前模块位于 `risk_project/features/tax.py`：

```python
# 绝对导入：从顶层包开始，含义最清楚
from risk_project.utils.time import parse_date

# 相对导入：一个点表示当前包，两个点表示上一级包
from ..utils.time import parse_date
```

建议：业务项目优先使用清晰的绝对导入；包内部很短、关系稳定时可使用相对导入。不要直接运行一个依赖相对导入的深层文件，应该从项目根目录使用 `python -m 包.模块`。


## D05. 导入只执行一次与 notebook 的旧代码问题

同一 Python 进程中，模块第一次导入后通常会缓存到 `sys.modules`，再次 `import` 不会自动重新执行文件。

这就是 notebook 中常见的情况：你修改了 `.py` 文件，但重新运行 `import module` 后好像没有变化。

学习时可以重启内核，或临时使用：

```python
import importlib
import my_module

importlib.reload(my_module)
```

正式程序不应依赖频繁 `reload`。


## D06. 常见导入错误

| 错误 | 常见原因 | 优先检查 |
|---|---|---|
| `ModuleNotFoundError` | 模块不在当前环境或路径中 | 解释器、虚拟环境、安装状态、工作目录 |
| `ImportError` | 模块存在，但目标名称不存在 | 拼写、版本、循环导入 |
| 导入了错误文件 | 自己的文件与标准库同名 | 是否命名为 `json.py`、`random.py`、`typing.py` |
| 相对导入失败 | 把包内深层模块当脚本直接运行 | 改用 `python -m package.module` |
| notebook 代码不更新 | 模块已缓存 | 重启内核或开发时 `reload` |

新手最容易忽略：**终端里的 Python 和 notebook 内核可能不是同一个环境。** 先打印 `sys.executable`。


# E. Python 的输入与输出全景

## E01. 先区分“数据从哪里来”

程序输入不只来自键盘：

| 输入来源 | 常用方式 | 场景 |
|---|---|---|
| 函数调用者 | 函数参数 | LeetCode、模块复用、测试 |
| 标准输入 | `input()`、`sys.stdin` | ACM 笔试、评测系统 |
| 命令行参数 | `argparse` | 训练脚本、批处理工具 |
| 环境变量 | `os.environ` | 密钥、部署配置 |
| 文件 | `open()`、`json`、`csv` | 本地数据与配置 |
| 数据库/API | 对应客户端库 | 生产系统 |

优秀代码会把“读取数据”和“处理数据”分开，使算法函数不依赖某一种输入来源。


## E02. 函数参数：最适合核心算法

```python
def two_sum(nums, target):
    ...
```

`nums` 和 `target` 由调用者传进来。函数不关心数据来自键盘、文件还是数据库，因此最容易测试和复用。


In [ ]:
def two_sum(nums, target):
    seen = {}
    for index, value in enumerate(nums):
        need = target - value
        if need in seen:
            return [seen[need], index]
        seen[value] = index
    return []


print(two_sum([2, 7, 11, 15], 9))


## E03. 标准输入：`input()` 与 `sys.stdin`

```python
name = input()                              # 读取一行，自动去掉末尾换行
n = int(input())                           # 字符串转整数
a, b = map(int, input().split())           # 一行两个整数
nums = list(map(int, input().split()))      # 一行整数数组
```

数据较多时：

```python
import sys
line = sys.stdin.readline().strip()
data = sys.stdin.buffer.read().split()
```

- `readline()` 每次读一行。
- `read()` 一次读完剩余内容。
- `buffer.read()` 得到字节，适合大量纯数字输入；转换时 `int(b'123')` 可以直接得到 `123`。


In [ ]:
# 用字符串模拟标准输入，避免这个代码格等待键盘输入
from io import StringIO

fake_stdin = StringIO("5\n1 2 3 4 5\n")
n = int(fake_stdin.readline())
nums = list(map(int, fake_stdin.readline().split()))

print("n =", n)
print("nums =", nums)


## E04. 命令行参数：`argparse`

算法训练项目经常这样运行：

```powershell
python -m risk_project.train --epochs 20 --seed 42
```

不要手动用下标大量解析 `sys.argv`，标准库 `argparse` 能自动处理类型、默认值和帮助信息。


In [ ]:
import argparse

parser = argparse.ArgumentParser(description="训练一个示例模型")
parser.add_argument("--epochs", type=int, default=10)
parser.add_argument("--seed", type=int, default=42)

# Notebook 中演示时手动传列表；真实脚本使用 parser.parse_args()
args = parser.parse_args(["--epochs", "20", "--seed", "7"])
print(args)


## E05. 文件输入：文本、JSON 与 CSV

```python
from pathlib import Path
text = Path('README.md').read_text(encoding='utf-8')
```

```python
import json
with open('config.json', encoding='utf-8') as file:
    config = json.load(file)
```

```python
import csv
with open('data.csv', newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
```

原则：明确编码；使用 `with` 管理文件；JSON/CSV 使用解析器，不要自己用字符串切割模拟格式解析。


In [ ]:
import json
import uuid
from pathlib import Path

config = {"model": "logistic_regression", "seed": 42}

path = Path(f".notebook_config_{uuid.uuid4().hex}.json")
try:
    path.write_text(json.dumps(config, ensure_ascii=False), encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
finally:
    path.unlink(missing_ok=True)

print(loaded)


## E06. 环境变量：适合部署差异和敏感配置入口

```python
import os

database_url = os.environ.get('DATABASE_URL')
debug = os.environ.get('DEBUG', 'false').lower() == 'true'
```

不要把真实密码、Token、数据库连接串提交进 Git。环境变量只是传入秘密的一种方式；正式团队通常还会使用密钥管理服务。


## E07. 输出也分不同目的

| 输出目的 | 推荐方式 | 说明 |
|---|---|---|
| 把结果交给调用者 | `return result` | 核心函数首选 |
| 给评测系统答案 | `print(...)` | 严格按题目格式 |
| 正常命令行结果 | `stdout` / `print` | 可被管道接收 |
| 错误和诊断 | `stderr` / `logging` | 不污染正常结果 |
| 持久化结果 | JSON、CSV、数据库、模型文件 | 明确格式与路径 |

`return` 和 `print` 不一样：`return` 把 Python 对象交回调用者；`print` 把文字写到输出流。


In [ ]:
def square(number):
    result = number * number
    print("调试信息：result =", result)
    return result


answer = square(5)
print("函数真正返回的值：", answer)


## E08. `stdout`、`stderr` 与日志

ACM 评测通常只比较标准输出，所以不要在答案中夹杂“请输入数字”“调试结果”等文字。

工程项目中，诊断信息应使用日志：

```python
import logging

logger = logging.getLogger(__name__)
logger.info('开始训练')
logger.warning('缺失率过高')
```

日志比到处 `print` 更适合工程，因为它有级别、时间、模块名和统一格式，并可输出到不同位置。


In [ ]:
import sys

print("正常结果写到 stdout")
print("诊断信息写到 stderr", file=sys.stderr)


# F. 大厂笔试和面试中的代码形式

## F01. 先说结论：没有全公司统一的一种模式

同一家公司在不同岗位、不同年份、不同轮次也可能使用不同平台。你应该同时准备下面四种形式：

1. **核心代码模式**：平台给函数或类，只补函数体。
2. **ACM 完整程序模式**：自己读取标准输入、调用算法、按格式打印。
3. **在线面试编辑器 / 共享 IDE**：可能允许运行，也可能要求你自己写测试。
4. **白板或文档手写**：更看重思路、边界、复杂度和沟通，不一定真的执行。

牛客企业端官方说明中，编程题明确存在“ACM 编程题模式”和“完善核心代码模式”两类。因此不要只会 LeetCode 函数体。


## F02. 核心代码模式（LeetCode 风格）

平台负责：

- 构造输入。
- 调用你的函数。
- 对比返回值。
- 某些题还会提供 `ListNode`、`TreeNode` 等定义。

你通常只写：

```python
class Solution:
    def twoSum(self, nums, target):
        seen = {}
        for i, value in enumerate(nums):
            need = target - value
            if need in seen:
                return [seen[need], i]
            seen[value] = i
        return []
```

不要擅自加 `input()`；是否需要 `class Solution`、类型注解和节点定义，以平台给出的模板为准。


## F03. ACM 完整程序模式

你负责完整流程：

```text
标准输入 -> 解析 -> 算法函数 -> 格式化 -> 标准输出
```

推荐结构：

```python
import sys

def solve(nums, target):
    # 纯算法逻辑
    ...

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    # 按题目格式解析 data
    answer = solve(...)
    print(...)

if __name__ == '__main__':
    main()
```

笔试时函数名不一定叫 `solve`，但把算法和输入输出分开非常有用。


## F04. ACM 高频输入模板

### 一行两个整数

```python
a, b = map(int, input().split())
print(a + b)
```

### 先给数量，再给数组

```python
n = int(input())
nums = list(map(int, input().split()))
assert len(nums) == n  # 正式提交可不写
```

### `n` 行矩阵

```python
n, m = map(int, input().split())
matrix = [list(map(int, input().split())) for _ in range(n)]
```

### 字符串中可能包含空格

```python
s = input().rstrip('\n')
```


## F05. 多组数据模板

### 第一行是测试组数 `T`

```python
t = int(input())
for _ in range(t):
    a, b = map(int, input().split())
    print(a + b)
```

### 读到 EOF（文件结束）

```python
import sys

for line in sys.stdin:
    if not line.strip():
        continue
    a, b = map(int, line.split())
    print(a + b)
```

### 一次读完所有空白分隔数据

```python
import sys

tokens = sys.stdin.buffer.read().split()
numbers = list(map(int, tokens))
```

究竟使用哪种，完全由题目的“输入描述”决定，不能看到多行就猜。


## F06. 输出格式高频模板

```python
print(answer)                         # 一个值
print(*nums)                          # 1 2 3
print(' '.join(map(str, nums)))       # 1 2 3
print('\n'.join(map(str, answers)))  # 每个答案一行
```

最常见的丢分原因不是算法，而是：

- 多打印提示词或调试文字。
- 需要空格分隔却直接打印列表，输出成 `[1, 2, 3]`。
- 漏掉多组数据。
- 把输入行数、数组长度、测试组数理解错。


In [ ]:
# 完整模拟一道 ACM 题：第一行 n 和 target，第二行 n 个整数
from io import StringIO

sample_input = StringIO("4 9\n2 7 11 15\n")


def find_pair(nums, target):
    seen = {}
    for index, value in enumerate(nums):
        need = target - value
        if need in seen:
            return seen[need], index
        seen[value] = index
    return -1, -1


def run_with_stream(stream):
    n, target = map(int, stream.readline().split())
    nums = list(map(int, stream.readline().split()))
    if len(nums) != n:
        raise ValueError("数组长度与 n 不一致")
    left, right = find_pair(nums, target)
    return f"{left} {right}"


print(run_with_stream(sample_input))


## F07. 链表和二叉树在面试中怎么输入

### 核心代码模式

平台常常已经定义节点并完成建表，你只接收 `head` 或 `root`。不要重复读取标准输入。

### ACM 或面试官要求自己测试

题目必须约定一种序列化方式，例如：

- 链表：一行数组 `1 2 3 4`。
- 二叉树：层序数组 `3 9 20 null null 15 7`。

你需要写“数组转结构”的辅助函数。面试时先问清楚输入表示，不要自己默默假设。


## F08. 面试现场推荐步骤

1. 复述问题，确认输入、输出、重复值、空输入和规模。
2. 先说暴力解法与复杂度，再提出优化方向。
3. 写核心函数，使用有意义的变量名。
4. 用最小样例手动走一遍。
5. 主动补边界：空、一个元素、重复、极值、无解。
6. 说明时间复杂度和空间复杂度。
7. 如果允许运行，先跑小样例，再修正错误。

面试官通常不只看最终答案，还看你如何澄清、拆解、测试和解释。


## F09. 30 秒判断该写哪一种

- 页面给了 `class Solution` 和函数签名：写核心代码，不写 `input()`。
- 页面有“输入描述 / 输出描述”：大概率写 ACM 完整程序。
- 面试官只口述问题：先问“需要完整输入输出，还是实现函数即可？”
- 面试官让你在自己的 IDE 写：建议写核心函数，并补 3 至 5 个测试调用；是否需要键盘输入先确认。
- SQL 题、机器学习设计题、数据分析题有各自形式，不要强行套算法题 ACM 模板。


# G. `solve()`、`main()` 与核心逻辑怎样分工

## G01. 三层结构

一个清晰的程序通常分成：

1. **输入适配层**：把文本、文件或参数变成 Python 对象。
2. **核心逻辑层**：接收对象，计算并返回对象。
3. **输出适配层**：把结果变成题目或用户需要的格式。

```text
input text -> parse_input() -> solve() -> format_output() -> print
```

好处：核心逻辑可以直接单元测试，也可以被 notebook、API、命令行共同调用。


In [ ]:
def parse_input(text):
    '''文本 -> Python 数据。'''
    lines = text.strip().splitlines()
    target = int(lines[0])
    nums = list(map(int, lines[1].split()))
    return nums, target


def solve(nums, target):
    '''Python 数据 -> Python 结果。'''
    return [value for value in nums if value >= target]


def format_output(values):
    '''Python 结果 -> 输出文本。'''
    return " ".join(map(str, values))


nums, target = parse_input("5\n1 5 3 8")
answer = solve(nums, target)
print(format_output(answer))


## G02. 什么时候叫 `solve()`，什么时候叫 `main()`

没有强制规则，但可以保持一致：

- `solve(...)`：解决问题的核心逻辑，尽量不直接读键盘。
- `main()`：程序入口，负责参数、文件、输入输出和流程编排。
- `train(...)`：模型训练核心流程。
- `predict(...)`：预测。
- `evaluate(...)`：评估。

ACM 时间紧时，`solve()` 也常直接读取 `sys.stdin` 并打印，这是比赛中的实用折中；工程代码则更建议拆层。


## G03. 不推荐的“所有事写在一个文件顶层”

```python
import pandas as pd

df = pd.read_csv('data.csv')
# 清洗 100 行
# 训练 100 行
# 评估 50 行
# 保存模型
```

问题：导入就开跑、难测试、难复用、配置散落、失败后难定位。

推荐：

```python
def load_data(config): ...
def build_features(data, config): ...
def train_model(features, config): ...
def evaluate(model, features): ...

def main():
    config = load_config()
    data = load_data(config)
    features = build_features(data, config)
    model = train_model(features, config)
    evaluate(model, features)

if __name__ == '__main__':
    main()
```


# H. 优秀算法工程师的 Python 项目是什么样的

## H01. 先看判断标准，不是目录越多越专业

一个优秀项目应该做到：

- **能运行**：新人按 README 可以复现。
- **能测试**：核心逻辑不依赖人工操作。
- **能复现**：依赖、配置、随机种子、数据版本和模型产物有记录。
- **能维护**：模块职责清楚，修改一处不会牵动所有文件。
- **能排错**：日志、异常和运行产物可追踪。
- **能交接**：别人知道入口、数据、命令、结果和限制。

小项目不需要为了“看起来专业”创建几十个空目录。结构随复杂度增长。


## H02. 刷题仓库的推荐结构

```text
leetcode-project/
├── README.md
├── pyproject.toml              # 可选：工具与依赖配置
├── solutions/
│   ├── __init__.py
│   ├── arrays.py
│   ├── linked_lists.py
│   └── trees.py
├── tests/
│   ├── test_arrays.py
│   └── test_linked_lists.py
├── notebooks/
│   └── learning_notes.ipynb
└── .gitignore
```

你的当前仓库以每日 notebook 学习为主，不必立刻重构。等你开始积累可复用答案和自动测试，再增加 `solutions/` 与 `tests/`。


## H03. 算法 / 机器学习工程项目的推荐结构

```text
risk-model/
├── README.md
├── pyproject.toml
├── .gitignore
├── configs/
│   ├── baseline.toml
│   └── production.toml
├── src/
│   └── risk_model/
│       ├── __init__.py
│       ├── __main__.py
│       ├── cli.py
│       ├── config.py
│       ├── data.py
│       ├── features.py
│       ├── train.py
│       ├── evaluate.py
│       └── inference.py
├── tests/
│   ├── test_features.py
│   └── test_inference.py
├── notebooks/
│   ├── 01_eda.ipynb
│   └── 02_error_analysis.ipynb
├── scripts/
│   └── download_sample_data.py
├── data/
│   └── README.md
├── artifacts/
│   └── .gitkeep
└── docs/
    └── model_card.md
```

这是一张“成熟项目地图”，不是要求每个练习项目全部照搬。


## H04. 各目录负责什么

| 位置 | 职责 | 不应该放什么 |
|---|---|---|
| `src/risk_model/` | 可导入、可测试的正式源码 | 临时实验和大数据文件 |
| `tests/` | 自动测试 | 人工观察才知道对错的代码 |
| `notebooks/` | EDA、实验、图表、讲解 | 唯一版本的核心业务逻辑 |
| `configs/` | 可审查的运行参数 | 密码和 Token |
| `scripts/` | 下载、迁移等辅助任务 | 核心模型逻辑复制品 |
| `data/` | 数据说明或小样例 | 大型敏感原始数据直接入 Git |
| `artifacts/` | 模型、指标、图表等运行产物 | 手写源码 |
| `docs/` | 设计、模型卡、数据说明 | 与代码事实冲突的旧说明 |


## H05. 为什么成熟项目常用 `src/` 布局

`src/` 把可导入源码与仓库根目录的脚本、配置、notebook 分开，可以减少“因为当前目录碰巧在搜索路径中，所以本地能导入，安装后却失败”的问题。

典型开发流程：

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install -e .
python -m pytest
python -m risk_model --config configs/baseline.toml
```

`pip install -e .` 表示可编辑安装：修改源码后通常无需重复安装。学习单文件脚本时不必使用 `src/`，多人项目或准备作品集时值得采用。


## H06. `pyproject.toml` 管什么

现代 Python 项目通常用它统一声明项目信息、Python 版本、依赖、构建方式和工具配置。

```toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "risk-model"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = []

[project.optional-dependencies]
dev = ["pytest", "ruff"]

[project.scripts]
risk-model = "risk_model.cli:main"

[tool.pytest.ini_options]
testpaths = ["tests"]
```

真实项目应固定或锁定依赖版本，并在 README 写清安装命令。


## H07. 一份合格的 `README.md`

至少回答：

1. 项目解决什么问题。
2. 环境和 Python 版本是什么。
3. 如何安装。
4. 最小运行命令是什么。
5. 输入数据格式是什么。
6. 如何运行测试。
7. 输出保存在哪里。
8. 当前效果、限制和已知问题是什么。

面试官克隆你的项目后，如果十分钟仍不知道怎么运行，项目价值会明显打折。


## H08. 算法工程代码的五条底线

### 1. 核心逻辑写成函数

便于测试、复用和组合。

### 2. 输入输出放在边界

核心函数接收 Python 对象并返回 Python 对象。

### 3. 配置不要散落在代码中

路径、阈值、随机种子、模型参数集中管理。

### 4. 错误要尽早暴露

检查数据列、维度、空值范围和参数合法性，不要默默产出错误结果。

### 5. 实验必须可追踪

至少记录代码版本、配置、数据版本、随机种子、指标和产物路径。


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class TrainConfig:
    learning_rate: float = 0.01
    epochs: int = 20
    seed: int = 42

    def validate(self):
        if self.learning_rate <= 0:
            raise ValueError("learning_rate 必须大于 0")
        if self.epochs <= 0:
            raise ValueError("epochs 必须大于 0")


config = TrainConfig()
config.validate()
print(config)


## H09. 测试应该长什么样

测试关注可观察行为，不是重复实现算法：

```python
from risk_model.features import normalize

def test_normalize_regular_values():
    assert normalize([0, 5, 10]) == [0.0, 0.5, 1.0]

def test_normalize_equal_values():
    assert normalize([3, 3]) == [0.0, 0.0]

def test_normalize_empty_input():
    assert normalize([]) == []
```

至少覆盖：正常情况、边界情况、异常情况。对于模型项目，还要测试数据 schema、特征列顺序、训练和推理的一致性。


## H10. Notebook 在算法项目中的正确位置

Notebook 很适合：

- 探索数据。
- 快速实验。
- 错误分析。
- 展示图表和结论。

Notebook 不适合成为：

- 唯一的数据清洗实现。
- 唯一的模型推理代码。
- 必须按神秘顺序运行才能复现的生产流程。

成熟过程：先在 notebook 探索，稳定后把函数移到 `src/`，给它写测试，再从 notebook 导入。


## H11. 文件命名和常见反例

推荐：

- 小写字母与下划线：`feature_engineering.py`
- 名称表达职责：`data_loader.py`、`metrics.py`
- 测试对应源码：`test_metrics.py`

避免：

- `test.py`、`new.py`、`final_final.py`、`utils2.py`
- 与标准库/第三方库同名：`random.py`、`json.py`、`pandas.py`
- 一个万能 `utils.py` 装几千行无关函数
- 在文件名中用空格、中文和特殊符号制作正式可导入包

学习 notebook 使用中文名没问题；正式 Python 包建议英文 ASCII 名，减少跨平台和工具链问题。


## H12. 一个完整的小项目调用链

```text
用户在终端执行命令
        |
        v
__main__.py / console script
        |
        v
cli.py 解析命令行参数
        |
        v
train.py 编排数据、特征、模型、评估
        |
        +--> data.py
        +--> features.py
        +--> evaluate.py
        |
        v
保存模型、指标和日志
```

入口文件应薄，核心模块应可独立导入测试。这就是 `if __name__ == '__main__':` 从一行语法走向项目结构的意义。


## H13. 用临时目录模拟一个真正的多文件项目

下面的代码会创建：

```text
demo_project/
├── app.py
└── algorithms.py
```

然后分别测试直接运行和导入。运行结束后临时目录自动删除。


In [ ]:
import subprocess
import shutil
import sys
import uuid
from pathlib import Path

algorithms_source = '''
def max_subarray(nums):
    if not nums:
        raise ValueError("nums 不能为空")
    best = current = nums[0]
    for value in nums[1:]:
        current = max(value, current + value)
        best = max(best, current)
    return best
'''

app_source = '''
from algorithms import max_subarray

def main():
    nums = list(map(int, input().split()))
    print(max_subarray(nums))

if __name__ == "__main__":
    main()
'''

root = Path(f".notebook_project_{uuid.uuid4().hex}")
root.mkdir()
try:
    (root / "algorithms.py").write_text(algorithms_source, encoding="utf-8")
    (root / "app.py").write_text(app_source, encoding="utf-8")

    run_result = subprocess.run(
        [sys.executable, "app.py"],
        cwd=root,
        input="-2 1 -3 4 -1 2 1 -5 4\n",
        capture_output=True,
        text=True,
        check=True,
    )
    import_result = subprocess.run(
        [sys.executable, "-c", "import app; print('导入完成')"],
        cwd=root,
        capture_output=True,
        text=True,
        check=True,
    )
finally:
    shutil.rmtree(root)

print("直接运行 app.py 的答案：", run_result.stdout.strip())
print("导入 app 时：", import_result.stdout.strip())


# I. 专项练习

## I01. 判断输出

文件 `helper.py`：

```python
print('A', __name__)

def work():
    print('B')

if __name__ == '__main__':
    work()
```

请分别回答：

1. 执行 `python helper.py` 会输出什么？
2. 在另一个文件中执行 `import helper` 会输出什么？
3. 为什么导入时仍然会输出 `A`？


### I01 参考答案

直接运行：

```text
A __main__
B
```

导入：

```text
A helper
```

原因：`print('A', __name__)` 是顶层语句，直接运行和第一次导入都会执行；`work()` 的调用受到入口判断保护。


## I02. 把有副作用的脚本改好

原代码：

```python
def average(nums):
    return sum(nums) / len(nums)

nums = list(map(float, input().split()))
print(average(nums))
```

要求：

1. `average()` 可以被安全导入。
2. 直接运行文件时仍能从标准输入读取并输出。
3. 空列表时给出清晰错误。


In [ ]:
# 请先自己完成，再看下一格参考答案
def average(nums):
    # TODO: 空列表检查并返回平均值
    pass


def main():
    # TODO: 读取、调用、打印
    pass


# TODO: 写入口判断


### I02 参考答案

```python
def average(nums):
    if not nums:
        raise ValueError('nums 不能为空')
    return sum(nums) / len(nums)

def main():
    nums = list(map(float, input().split()))
    print(average(nums))

if __name__ == '__main__':
    main()
```


## I03. 把 LeetCode 形式改成 ACM 形式

题目约定：

- 第一行输入 `n target`。
- 第二行输入 `n` 个整数。
- 输出两个下标，用一个空格分隔；无解输出 `-1 -1`。

请把下面函数包装成完整程序：

```python
def two_sum(nums, target):
    ...
```


In [ ]:
# 练习：代码格中再次写出完整问题，方便独立完成
# 输入：第一行 n target；第二行 n 个整数
# 目标：找到和为 target 的两个元素下标
# 输出：两个下标，空格分隔；无解输出 -1 -1
# 注意：需要包含算法函数、输入解析、输出和入口判断

def two_sum(nums, target):
    pass


def main():
    pass


# 在这里补入口判断


### I03 参考答案

```python
def two_sum(nums, target):
    seen = {}
    for i, value in enumerate(nums):
        need = target - value
        if need in seen:
            return seen[need], i
        seen[value] = i
    return -1, -1

def main():
    n, target = map(int, input().split())
    nums = list(map(int, input().split()))
    if len(nums) != n:
        raise ValueError('数组长度与 n 不一致')
    left, right = two_sum(nums, target)
    print(left, right)

if __name__ == '__main__':
    main()
```


## I04. 项目结构判断

下面哪些做法需要改进？

1. `train.py` 被导入时立刻读取 5GB 数据并开始训练。
2. notebook 中探索出稳定特征后，把实现移到 `src/project/features.py` 并写测试。
3. 密码直接写在 `config.py` 并提交 Git。
4. `main()` 负责参数解析和流程编排，算法函数只接收数据并返回结果。
5. 项目同时存在 `utils.py`、`utils2.py`、`new_utils_final.py`。


### I04 参考答案

- 1 需要改：导入副作用严重，应把训练放入函数和入口。
- 2 合理：notebook 负责探索，稳定逻辑进入源码和测试。
- 3 需要改：秘密不应进入版本库。
- 4 合理：入口与核心逻辑职责清楚。
- 5 需要改：应按职责拆分并使用明确名称。


# J. 最终速查表

## J01. 最值得背下来的单文件模板

```python
import sys

def solve(nums):
    '''核心算法：接收 Python 对象，返回 Python 对象。'''
    return sum(nums)

def main():
    nums = list(map(int, sys.stdin.buffer.readline().split()))
    answer = solve(nums)
    print(answer)

if __name__ == '__main__':
    main()
```

口述：定义算法函数；定义输入输出入口；只有直接运行当前文件时才调用入口，因此该文件也能被安全导入。


## J02. 面试模式速查

| 看到什么 | 写什么 |
|---|---|
| `class Solution` / 给定函数签名 | 只实现核心函数 |
| 输入描述 + 输出描述 | 完整 ACM 程序 |
| 面试官口述 | 先确认需要函数还是完整程序 |
| 自己 IDE 共享屏幕 | 核心函数 + 主动测试，输入方式先确认 |
| 链表 / 树由平台提供节点 | 不重复定义，按模板实现 |
| 链表 / 树要求自己构造 | 先确认序列化格式，再写建表函数 |


## J03. 项目自检清单

- [ ] 导入模块不会自动开始训练、读输入或删除文件。
- [ ] 入口使用 `main()`，并由 `if __name__ == '__main__':` 控制。
- [ ] 核心算法接收参数并返回结果，不绑定键盘输入。
- [ ] README 有安装、运行、测试和输入数据说明。
- [ ] 依赖与 Python 版本有记录。
- [ ] 正式源码、测试、notebook、配置和产物职责分开。
- [ ] 有正常、边界和异常测试。
- [ ] 没有提交密码、大数据、缓存和临时产物。
- [ ] 随机种子、配置、数据版本和指标可以追踪。
- [ ] 文件名明确，不使用 `final_final.py` 一类名字。


## J04. 最小通关标准

不看答案完成下面五句话：

1. `__name__` 是 ______ 提供的变量，表示 ______。
2. 直接运行文件时，`__name__` 等于 ______。
3. 导入 `utils.py` 时，文件内 `__name__` 通常等于 ______。
4. `if __name__ == '__main__':` 的目的不是阻止导入，而是 ______。
5. ACM 与 LeetCode 核心代码模式的最大区别是 ______。

答案：Python；当前模块名；`'__main__'`；`'utils'`；只在当前模块作为入口时执行入口逻辑；ACM 需要自己处理标准输入和标准输出。


# K. 官方资料与继续学习

- [Python 官方文档：`__main__` 顶层代码环境](https://docs.python.org/3/library/__main__.html)
- [Python 官方教程：模块与包](https://docs.python.org/3/tutorial/modules.html)
- [Python Packaging User Guide：打包项目教程](https://packaging.python.org/en/latest/tutorials/packaging-projects/)
- [Python Packaging User Guide：`src` 布局与扁平布局](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/)
- [Python Packaging User Guide：编写 `pyproject.toml`](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/)
- [牛客企业端：ACM 编程题模式与完善核心代码模式](https://hr.nowcoder.com/article/416?urlSource=sitemap)

下一步实践顺序：先用 J01 模板独立写三道简单题的 ACM 版本，再给一个自己的算法函数补 `tests/`，最后尝试创建一个只有 `src/`、`tests/`、`README.md` 和 `pyproject.toml` 的小项目。
